# NETWORK MODELING FOR INFECTIOUS DISEASE SPREAD




– Data Preprocessing


## Data Import

In [ ]:
import pandas as pd
import numpy as np

# Loading Data
df = pd.read_csv("merged_cdc_census_bea_county_data.csv")

# Sorting Data
df = df.sort_values(["state", "county", "year", "MMWR WEEK"])

df.head()

,state,county,year,MMWR WEEK,Label,previous_52_week_max,latitude,longitude,population_estimate,income
26,ALABAMA,Chilton,2022,1,Anthrax,0,32.756889,-86.844516,45868.0,43434.0
72,ALABAMA,Chilton,2022,1,"Arboviral diseases, Chikungunya virus disease",0,32.756889,-86.844516,45868.0,43434.0
118,ALABAMA,Chilton,2022,1,"Arboviral diseases, Eastern equine encephaliti...",0,32.756889,-86.844516,45868.0,43434.0
164,ALABAMA,Chilton,2022,1,"Arboviral diseases, Jamestown Canyon virus di...",0,32.756889,-86.844516,45868.0,43434.0
210,ALABAMA,Chilton,2022,1,"Arboviral diseases, La Crosse virus disease",0,32.756889,-86.844516,45868.0,43434.0


In [ ]:
# Data Structure
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 725787 entries, 26 to 725782
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   state                 725787 non-null  object 
 1   county                725787 non-null  object 
 2   year                  725787 non-null  int64  
 3   MMWR WEEK             725787 non-null  int64  
 4   Label                 725787 non-null  object 
 5   previous_52_week_max  725787 non-null  int64  
 6   latitude              725787 non-null  float64
 7   longitude             725787 non-null  float64
 8   population_estimate   725787 non-null  float64
 9   income                725787 non-null  float64
dtypes: float64(4), int64(3), object(3)
memory usage: 60.9+ MB


In [ ]:
# Checking the number of infections under infection label
print(df['Label'].value_counts().to_string())

Label
Anthrax                                                                                                           7208
Arboviral diseases, Eastern equine encephalitis virus disease                                                     7208
Arboviral diseases, Western equine encephalitis virus disease                                                     7208
Arboviral diseases, Jamestown Canyon  virus disease                                                               7208
Arboviral diseases, La Crosse  virus disease                                                                      7208
Arboviral diseases, St. Louis encephalitis virus disease                                                          7208
Dengue virus infections, Dengue-like illness                                                                      7208
Poliomyelitis, paralytic                                                                                          7208
Poliovirus infection, nonparalytic        

In [ ]:
# Level Combination and Reduction
df['Label_grouped'] = df['Label'].str.split(',').str[0].str.strip()
# Rechecking levels
print(df['Label_grouped'].value_counts().to_string())

Label_grouped
Arboviral diseases                                                                         57646
Viral hemorrhagic fevers                                                                   48624
Meningococcal disease                                                                      35962
Haemophilus influenzae                                                                     35936
Invasive pneumococcal disease                                                              24465
Dengue virus infections                                                                    21603
Botulism                                                                                   21583
Q fever                                                                                    21400
Hepatitis C                                                                                21186
Ehrlichiosis and Anaplasmosis                                                              17546
Rubella         

In [ ]:
# Getting disease groups mapping
group_map = {
    # STI
    'Chlamydia trachomatis infection': 'STI',
    'Gonorrhea': 'STI',
    'Syphilis': 'STI',
    'Chancroid': 'STI',

    # Respiratory
    'Tuberculosis': 'Respiratory',
    'Pertussis': 'Respiratory',
    'Legionellosis': 'Respiratory',
    'Invasive pneumococcal disease': 'Respiratory',
    'Haemophilus influenzae': 'Respiratory',
    'Influenza-associated pediatric mortality': 'Respiratory',
    'Novel Influenza A virus infections': 'Respiratory',

    # Enteric / Foodborne
    'Campylobacteriosis': 'Enteric',
    'Salmonellosis (excluding Salmonella Typhi infection and Salmonella Paratyphi infection)': 'Enteric',
    'Shiga toxin-producing Escherichia coli (STEC)': 'Enteric',
    'Shigellosis': 'Enteric',
    'Cryptosporidiosis': 'Enteric',
    'Giardiasis': 'Enteric',
    'Cyclosporiasis': 'Enteric',
    'Cholera': 'Enteric',
    'Botulism': 'Enteric',
    'Vibriosis (any species of the family Vibrionaceae': 'Enteric',
    'Hemolytic uremic syndrome post-diarrheal': 'Enteric',
    'Salmonella Typhi infection': 'Enteric',
    'Salmonella Paratyphi infection': 'Enteric',

    # Hepatitis
    'Hepatitis': 'Hepatitis',
    'Hepatitis A': 'Hepatitis',
    'Hepatitis B': 'Hepatitis',
    'Hepatitis C': 'Hepatitis',

    # Vector-borne
    'Arboviral diseases': 'Vector-borne',
    'Dengue virus infections': 'Vector-borne',
    'Babesiosis': 'Vector-borne',
    'Ehrlichiosis and Anaplasmosis': 'Vector-borne',
    'Malaria': 'Vector-borne',
    'Zika virus disease': 'Vector-borne',

    # Vaccine-preventable
    'Varicella disease': 'Vaccine-preventable',
    'Varicella morbidity': 'Vaccine-preventable',
    'Measles': 'Vaccine-preventable',
    'Mumps': 'Vaccine-preventable',
    'Rubella': 'Vaccine-preventable',
    'Poliomyelitis': 'Vaccine-preventable',
    'Tetanus': 'Vaccine-preventable',

    # Zoonotic
    'Rabies': 'Zoonotic',
    'Q fever': 'Zoonotic',
    'Tularemia': 'Zoonotic',
    'Brucellosis': 'Zoonotic',
    'Leptospirosis': 'Zoonotic',
    'Psittacosis': 'Zoonotic',
    'Anthrax': 'Zoonotic',
    'Plague': 'Zoonotic',

    # Healthcare-associated / AMR
    'Candida auris': 'Healthcare-associated',
    'Carbapenemase-producing carbapenem-resistant Enterobacteriaceae': 'Healthcare-associated',
    'Vancomycin-resistant Staphylococcus aureus': 'Healthcare-associated',
    'Vancomycin-intermediate Staphylococcus aureus': 'Healthcare-associated',

    # Invasive bacterial
    'Meningococcal disease': 'Invasive bacterial',
    'Listeriosis': 'Invasive bacterial',
    'Streptococcal toxic shock syndrome': 'Invasive bacterial',
    'Toxic shock syndrome (other than Streptococcal)': 'Invasive bacterial',

    # High consequence
    'Mpox': 'High-consequence',
    'Hantavirus pulmonary syndrome': 'High-consequence',
    'Hantavirus infection': 'High-consequence',
    'Melioidosis': 'High-consequence',

    # Fungal
    'Coccidioidomycosis': 'Fungal'
}

df['Disease_Group'] = (
    df['Label_grouped']
    .map(group_map)
    .fillna('Other')
)



In [ ]:
df_spread = df[df['previous_52_week_max'] > 0]
df_spread.info()

<class 'pandas.core.frame.DataFrame'>
Index: 333028 entries, 348 to 725243
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   state                 333028 non-null  object 
 1   county                333028 non-null  object 
 2   year                  333028 non-null  int64  
 3   MMWR WEEK             333028 non-null  int64  
 4   Label                 333028 non-null  object 
 5   previous_52_week_max  333028 non-null  int64  
 6   latitude              333028 non-null  float64
 7   longitude             333028 non-null  float64
 8   population_estimate   333028 non-null  float64
 9   income                333028 non-null  float64
 10  Label_grouped         333028 non-null  object 
 11  Disease_Group         333028 non-null  object 
dtypes: float64(4), int64(3), object(5)
memory usage: 33.0+ MB


In [ ]:
print(df_spread.groupby('Label_grouped')['previous_52_week_max'].sum().sort_values(ascending=False).to_string())

Label_grouped
Chlamydia trachomatis infection                                                            3764949
Gonorrhea                                                                                  1469817
Campylobacteriosis                                                                          377063
Salmonellosis (excluding Salmonella Typhi infection and Salmonella Paratyphi infection)     341854
Syphilis                                                                                    248196
Coccidioidomycosis                                                                          177310
Invasive pneumococcal disease                                                               174550
Shiga toxin-producing Escherichia coli (STEC)                                               122750
Shigellosis                                                                                 108671
Cryptosporidiosis                                                                            90

In [ ]:
# Filtering data for most common infections of > 50000 cases
disease_totals = df_spread.groupby('Label_grouped')['previous_52_week_max'].sum().sort_values(ascending=False)
high_burden = disease_totals[disease_totals > 50000]

print(high_burden)

Label_grouped
Chlamydia trachomatis infection                                                            3764949
Gonorrhea                                                                                  1469817
Campylobacteriosis                                                                          377063
Salmonellosis (excluding Salmonella Typhi infection and Salmonella Paratyphi infection)     341854
Syphilis                                                                                    248196
Coccidioidomycosis                                                                          177310
Invasive pneumococcal disease                                                               174550
Shiga toxin-producing Escherichia coli (STEC)                                               122750
Shigellosis                                                                                 108671
Cryptosporidiosis                                                                            90

In [ ]:
keep_labels = high_burden.index

df_high_burden = df_spread[
    df_spread['Label_grouped'].isin(keep_labels)
]

In [ ]:
df_high_burden.info()

<class 'pandas.core.frame.DataFrame'>
Index: 174042 entries, 348 to 724997
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   state                 174042 non-null  object 
 1   county                174042 non-null  object 
 2   year                  174042 non-null  int64  
 3   MMWR WEEK             174042 non-null  int64  
 4   Label                 174042 non-null  object 
 5   previous_52_week_max  174042 non-null  int64  
 6   latitude              174042 non-null  float64
 7   longitude             174042 non-null  float64
 8   population_estimate   174042 non-null  float64
 9   income                174042 non-null  float64
 10  Label_grouped         174042 non-null  object 
 11  Disease_Group         174042 non-null  object 
dtypes: float64(4), int64(3), object(5)
memory usage: 17.3+ MB


In [ ]:
# Saving high burden data
df_high_burden.to_csv('df_high_burden.csv', index=False)